# Stoic Qwen3.8 27B DPO Training with Unsloth (4-bit QLoRA)

**Phase 2:** Direct Preference Optimization continuing the Qwen3.8 27B v2 SFT LoRA.

DPO sharpens persona behavior by contrasting preferred persona-voiced answers with generic, shallow, or out-of-voice answers.

**Base Model:** `unsloth/Qwen3.8-27B` — bf16 weights (`Qwen3_5ForConditionalGeneration`, `model_type` `qwen3_5`), quantized to bnb NF4 on the fly. Multimodal hybrid: 48 gated-delta `linear_attn` + 16 `self_attn` language layers, plus a vision tower and a 1-layer MTP head.

**Starting Adapter:** `output/stoic_qwen38_27b_unsloth_4bit/lora_adapters` — produced by `stoic_qwen38_27b_sft.ipynb`. This notebook resolves that path from `SFT_MODEL_NAME_BASE`, which must match the SFT notebook's `MODEL_NAME_BASE` exactly.

**Dataset:** v2 DPO preference pairs from the Stoic persona datagen pipeline (1,800 pairs across 3 rejection sources).

**Training Hardware:** NVIDIA DGX Spark (GB10, sm_120, 128 GB unified memory).

**Chat Template:** Qwen ChatML via `tokenizer.apply_chat_template`.

**Thinking mode:** prompt formatting always uses `enable_thinking=False` so Qwen3.8's default reasoning instruction never enters the preference prompts. `ENABLE_THINKING` controls only the eval cells and must match what SFT trained on.

**LoRA scope:** this notebook does NOT call `get_peft_model`. It continues the SFT adapter, so it inherits that adapter's module set. A leaked SFT adapter stays leaked — the model-loading cell asserts the scope before any expensive work starts. See `docs/multimodal_and_hybrid_base_models.md`.

## 1. Configuration

All paths, hyperparameters, and run-shaping values for this notebook.

In [ ]:
import os
from pathlib import Path

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
# Normally run inside the `unsloth-notebook` container, which bind-mounts
# /home/spark/projects/training -> /workspace/training.
if os.path.exists("/workspace/training/stoic"):
    PROJECT_ROOT = Path("/workspace/training/stoic")
elif os.path.exists("/workspace/stoic"):
    PROJECT_ROOT = Path("/workspace/stoic")
else:
    PROJECT_ROOT = Path("/home/spark/projects/training/stoic")

OUTPUT_ROOT = PROJECT_ROOT / "output"

# =========================== HUGGING FACE CACHE ===========================
# Use the default Hub cache: /root/.cache/huggingface/hub in the unsloth-notebook container,
# which is bind-mounted from /home/spark/.cache/huggingface/hub on the host. Do NOT override
# HF_HUB_CACHE: pointing it at the cache root (not hub/) made unsloth build a second
# models--* tree beside hub/ and re-download every base model.

# =========================== DGX SPARK ALLOCATOR ===========================
# GB10 unified memory: force CUDA allocator reclamation instead of letting PyTorch treat
# all 128 GB as endlessly cacheable CUDA memory. Must be set before CUDA-heavy imports.
#
# NOTE: no torch.cuda.set_per_process_memory_fraction() here, deliberately. This notebook
# has a GGUF merge stage that reloads the base at load_in_4bit=False (~56 GB of bf16
# weights); a 0.55 fraction caps the process at ~70 GB and the merge can trip over it.
# Periodic gc + empty_cache during training (see the trainer cell) does the real work.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.5,max_split_size_mb:256"

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM = "unsloth/Qwen3.8-27B"
MODEL_NAME_BASE = "stoic_qwen38_27b_unsloth_4bit_dpo"

# MUST match MODEL_NAME_BASE in stoic_qwen38_27b_sft.ipynb. This is the SFT -> DPO contract.
SFT_MODEL_NAME_BASE = "stoic_qwen38_27b_unsloth_4bit"
SFT_LORA_PATH = OUTPUT_ROOT / SFT_MODEL_NAME_BASE / "lora_adapters"

# =========================== THINKING MODE ===========================
# Prompt formatting for DPO ALWAYS uses enable_thinking=False (see the dataset cell) so
# Qwen3.8's default xhigh reasoning instruction never contaminates the preference prompts.
# This flag controls only the eval cells, and must match what the SFT run trained on.
ENABLE_THINKING = False

# =========================== INPUT DATA ===========================
DPO_DATA_FILE = PROJECT_ROOT / "data/training-data/stoic_persona/stoic_personas_dpo.jsonl"

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = OUTPUT_ROOT / MODEL_NAME_BASE
TRAIN_DIR = OUTPUT_BASE_DIR / "train"
LORA_OUTPUT_DIR = OUTPUT_BASE_DIR / "lora_adapters"

# =========================== DPO HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 4096
MAX_PROMPT_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 5e-6
TARGET_EPOCHS = 1
DPO_BETA = 0.1
LOSS_TYPE = "sigmoid"
WARMUP_RATIO = 0.1
DPO_MAX_PAIRS = 0  # 0 = use all pairs; set e.g. 3000/6000 for capped runs, stratified by source

# =========================== CHECKPOINTING / MEMORY ===========================
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3
# gc.collect() + torch.cuda.empty_cache() cadence. Every 50 steps, NOT every step -
# per-step cache flushing measurably slows training for no additional benefit.
CLEANUP_STEPS = 50

# =========================== EVAL ===========================
# Defaults from the base model's own generation_config.json.
TEST_PROMPT = "I am struggling with anxiety about things outside my control. How do I find peace?"
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95
GEN_TOP_K = 20

# ============================================================================
print("=" * 60)
print("CONFIGURATION LOADED (Qwen3.8 27B DPO)")
print("=" * 60)
print(f"  Project root:  {PROJECT_ROOT}")
print(f"  HF hub cache:  {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"  Base model:    {BASE_LLM}")
print(f"  SFT LoRA:      {SFT_LORA_PATH}")
print(f"  DPO data:      {DPO_DATA_FILE}")
print(f"  Output:        {OUTPUT_BASE_DIR}")
print(f"  Training:      batch={BATCH_SIZE} x {GRAD_ACCUM} (effective {BATCH_SIZE*GRAD_ACCUM}), "
      f"lr={LEARNING_RATE}, beta={DPO_BETA}, loss={LOSS_TYPE}")
print(f"  DPO max pairs: {DPO_MAX_PAIRS or 'ALL (no limit)'}")
print(f"  Checkpoints:   every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Thinking mode: {'ON' if ENABLE_THINKING else 'OFF'} (eval cells only)")

# Fail here, not three cells and a 56 GB download later.
if not DPO_DATA_FILE.exists():
    raise FileNotFoundError(f"DPO data not found: {DPO_DATA_FILE}")
if not SFT_LORA_PATH.exists():
    raise FileNotFoundError(
        f"SFT LoRA not found at {SFT_LORA_PATH}. Run stoic_qwen38_27b_sft.ipynb first, "
        f"and confirm SFT_MODEL_NAME_BASE here matches MODEL_NAME_BASE there."
    )
print(f"\n  Input data and SFT adapter both present.")

## 2. Environment Preparation

Safe to re-run. Order matters:

1. Verify the CUDA PyTorch build is intact (no side effects).
2. Core training packages, and remove the aarch64 `torchao` build that blocks PEFT's bnb 4-bit dispatcher.
3. `transformers` + `peft` from git main — the `qwen3_5` architecture is newer than the container's release.
4. Small utility packages.
5. Rebuild `causal_conv1d` from source — the NGC image ships the Python package **without** its compiled CUDA extension, which hard-crashes any import that touches the FalconH1 model inside `transformers`/`unsloth`.
6. `flash-linear-attention` — Triton kernels for the Qwen3.8 gated-delta-rule fast path.
7. Import `unsloth` **before** `transformers` so its monkey-patches apply.

**Restart the kernel after this cell, then resume from Cell 1.**

The previous revision of this notebook did installs, `import torch`, and CUDA calls in a single cell with no restart instruction. That leaves stale modules loaded against freshly installed packages.

In [ ]:
import os, sys, subprocess, importlib, importlib.util

def _pip(*args, env_extra=None):
    """Run pip against this kernel's interpreter; print output only on failure."""
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# --- 1. Verify the CUDA PyTorch build is intact ------------------------------
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "cpu" in torch.__version__:
        print("  CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through, or another container holds it.")
        print("  Check `docker ps` for a running sglang-* / vllm-* server and stop it.")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")
print(f"  torch {torch.__version__} - CUDA {torch.version.cuda} - GPU: {torch.cuda.get_device_name(0)}")

# --- 2. Core training packages ------------------------------------------------
print("  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...")
_pip("install", "-q", "-U", "unsloth", "trl", "accelerate", "datasets", "bitsandbytes")

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel >=0.16 on PyPI, so
# uninstall - peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
_pip("uninstall", "-y", "-q", "torchao")

# --- 3. transformers + peft from git main -------------------------------------
print("  Installing transformers + peft from git main...")
_pip("install", "-q", "-U", "git+https://github.com/huggingface/transformers.git")
_pip("install", "-q", "-U", "git+https://github.com/huggingface/peft.git")

# --- 4. Small utility packages ------------------------------------------------
for _module, _install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(_module) is None:
        print(f"  Installing {_install_args[-1]}...")
        _pip(*_install_args)

# --- 5. Fix causal_conv1d -----------------------------------------------------
# The NGC image ships causal_conv1d WITHOUT its compiled CUDA extension, which
# hard-crashes any import reaching the FalconH1 model inside transformers/unsloth.
# pip also caches a broken prebuilt aarch64 wheel, so --no-binary is required.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",   # DGX Spark GB10 = sm_120
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing - rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    _ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
               "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if _ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext - uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed - uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# --- 6. flash-linear-attention ------------------------------------------------
if _check_import("fla") is None:
    print("  Installing flash-linear-attention...")
    _pip("install", "-q", "--no-deps", "flash-linear-attention", "fla-core")

_fast_path_ok = False
try:
    from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
    _fast_path_ok = _causal_ok and chunk_gated_delta_rule is not None
    for _k in list(sys.modules.keys()):
        if _k.startswith("fla."):
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError):
    pass
print(f"  Fast path: {'ENABLED' if _fast_path_ok else 'DISABLED (using torch fallback)'}")

# --- 7. Import unsloth FIRST, then transformers -------------------------------
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
import peft
import trl

print()
print(f"  unsloth:       {unsloth.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  trl:           {trl.__version__}")
print(f"  causal_conv1d: {'OK' if _causal_ok else 'FALLBACK (torch path)'}")
print(f"  fla:           {'OK' if _check_import('fla') else 'MISSING'}")
print(f"  torchao:       {'PRESENT (should be absent)' if importlib.util.find_spec('torchao') else 'absent (correct)'}")
print()
print("Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).")

## 3. Load DPO Dataset

Load the preference pairs produced by the Stoic datagen DPO notebook. Expected format: `{chosen: [...messages], rejected: [...messages], source: str, persona: str}`.

In [ ]:
import json
from collections import Counter, defaultdict
from datasets import Dataset as HFDataset

if not DPO_DATA_FILE.exists():
    raise FileNotFoundError(f"DPO data not found: {DPO_DATA_FILE}")

with open(DPO_DATA_FILE) as f:
    raw_pairs = [json.loads(line) for line in f if line.strip()]

source_counts = Counter(pair.get("source", "unknown") for pair in raw_pairs)
persona_counts = Counter(pair.get("persona", "unknown") for pair in raw_pairs)
persona_system_prompts = {}

for pair in raw_pairs:
    persona = pair.get("persona", "unknown")
    chosen = pair.get("chosen", [])
    if chosen and chosen[0].get("role") == "system":
        persona_system_prompts.setdefault(persona, chosen[0].get("content", ""))

print(f"Loaded {len(raw_pairs)} DPO pairs from {DPO_DATA_FILE}")
print(f"Personas: {len(persona_counts)}")
print(f"System prompts extracted: {len(persona_system_prompts)}")

print("\nSource distribution:")
for source, count in source_counts.most_common():
    print(f"  {source:30s} {count:>6}")

print("\nTop personas:")
for persona, count in persona_counts.most_common(12):
    print(f"  {persona:30s} {count:>6}")

## 4. Load Model & Tokenizer (4-bit)

Load the SFT LoRA as the DPO starting point. This continues training the same adapter instead of stacking a new one, so the output stays a single LoRA against the Qwen3.8 27B base.

In [ ]:
import os
# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and the broken
# torch.compile path. Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

from unsloth import FastLanguageModel
import torch

print(f"Loading SFT LoRA from: {SFT_LORA_PATH}")
model, tokenizer = FastLanguageModel.from_pretrained(
    str(SFT_LORA_PATH),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Qwen3.8 27B is Qwen3_5ForConditionalGeneration, so Unsloth returns a multimodal
# Processor, not a raw tokenizer. Unwrap it: TRL then takes its text path instead of
# calling process_row(), which expects image inputs. This is the same pattern the
# PubMed Qwen3.6 27B notebooks use, and it is why the model_type swap in the trainer
# cell is now a belt-and-braces assertion rather than the only thing holding the line.
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor - text-only DPO mode)")
text_tokenizer = tokenizer   # backwards-compatible alias used by later cells

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
if getattr(model, "generation_config", None) is not None:
    model.generation_config.pad_token_id = tokenizer.pad_token_id

# ============ VERIFY THE INHERITED LoRA SCOPE ============
# DPO does NOT call get_peft_model - it continues the SFT adapter and inherits its module
# set exactly. If the SFT run leaked adapters into the vision tower or the MTP head, this
# run inherits the leak, wastes hours, and produces an adapter vLLM refuses to load.
# Catch it here, before the reference-logprob precompute.
_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
_stray = [n for n in _adapted
          if n.startswith("mtp.") or ".visual." in n
          or "vision_tower" in n or "audio_tower" in n]
if _stray:
    raise RuntimeError(
        f"The SFT adapter has {len(_stray)} LoRA module(s) outside the language model "
        f"(e.g. {_stray[:3]}). This cannot be fixed at DPO - it is inherited. Re-run SFT "
        "with finetune_vision_layers=False. See docs/multimodal_and_hybrid_base_models.md."
    )

print("Model + SFT LoRA loaded")
print(f"  Base:            {BASE_LLM}")
print(f"  Adapter:         {SFT_LORA_PATH}")
print(f"  Adapted modules: {len(_adapted)} (all inside the language model)")
print(f"  Precision:       4-bit QLoRA")
print(f"  Max seq len:     {MAX_SEQ_LENGTH}")
print(f"  Tokenizer:       {type(tokenizer).__name__}"
      f"{' (unwrapped from ' + type(processor).__name__ + ')' if processor else ''}")
print(f"  GPU alloc:       {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 5. Validate & Prepare DPO Dataset

Convert conversational `chosen` / `rejected` examples into the `{prompt, chosen, rejected}` text format expected by TRL's `DPOTrainer`.

Prompts are rendered with `enable_thinking=False` so Qwen3.8's default reasoning instruction never enters the preference prompt. This must match how the SFT data was formatted.

In [ ]:
import random

errors = []
formatted_pairs = []
skipped_too_long = 0

for i, pair in enumerate(raw_pairs):
    chosen_msgs = pair.get("chosen", [])
    rejected_msgs = pair.get("rejected", [])

    if len(chosen_msgs) < 3 or len(rejected_msgs) < 3:
        errors.append(f"Pair {i}: too few messages (chosen={len(chosen_msgs)}, rejected={len(rejected_msgs)})")
        continue

    if chosen_msgs[:-1] != rejected_msgs[:-1]:
        errors.append(f"Pair {i}: chosen/rejected prompt mismatch")
        continue

    if chosen_msgs[-1].get("role") != "assistant" or rejected_msgs[-1].get("role") != "assistant":
        errors.append(f"Pair {i}: final messages must both be assistant responses")
        continue

    prompt_msgs = chosen_msgs[:-1]
    chosen_response = chosen_msgs[-1].get("content", "")
    rejected_response = rejected_msgs[-1].get("content", "")

    if not chosen_response.strip() or not rejected_response.strip():
        errors.append(f"Pair {i}: empty chosen or rejected response")
        continue
    if chosen_response.strip() == rejected_response.strip():
        errors.append(f"Pair {i}: chosen and rejected responses are identical")
        continue

    prompt_kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, enable_thinking=False, **prompt_kwargs)
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, **prompt_kwargs)

    p_len = len(text_tokenizer.encode(prompt_text, add_special_tokens=False))
    c_len = len(text_tokenizer.encode(chosen_response, add_special_tokens=False))
    r_len = len(text_tokenizer.encode(rejected_response, add_special_tokens=False))

    if p_len > MAX_PROMPT_LENGTH or p_len + max(c_len, r_len) > MAX_SEQ_LENGTH:
        skipped_too_long += 1
        continue

    formatted_pairs.append({
        "prompt": prompt_text,
        "chosen": chosen_response,
        "rejected": rejected_response,
        "source": pair.get("source", "unknown"),
        "persona": pair.get("persona", "unknown"),
    })

if errors:
    print(f"WARNING: {len(errors)} validation errors")
    for error in errors[:10]:
        print(f"  {error}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more")
else:
    print(f"All {len(raw_pairs)} pairs passed validation")

if skipped_too_long:
    print(f"Filtered {skipped_too_long} pairs exceeding prompt/sequence limits")

# Stratified sampling — proportional by source, preserving distribution (matches the
# Biblical/PubMed DPO notebooks).
if DPO_MAX_PAIRS and len(formatted_pairs) > DPO_MAX_PAIRS:
    random.seed(3407)
    by_source = defaultdict(list)
    for pair in formatted_pairs:
        by_source[pair.get("source", "unknown")].append(pair)
    sampled = []
    total = len(formatted_pairs)
    for items in by_source.values():
        sample_count = max(1, round(DPO_MAX_PAIRS * len(items) / total))
        sampled.extend(random.sample(items, min(sample_count, len(items))))
    random.shuffle(sampled)
    formatted_pairs = sampled[:DPO_MAX_PAIRS]
    print(f"Capped formatted DPO dataset to {len(formatted_pairs):,} pairs (stratified by source)")

dpo_dataset = HFDataset.from_list(formatted_pairs).shuffle(seed=42)

prompt_lens = [len(p["prompt"]) for p in formatted_pairs]
chosen_lens = [len(p["chosen"]) for p in formatted_pairs]
rejected_lens = [len(p["rejected"]) for p in formatted_pairs]

print(f"\nFormatted DPO dataset: {len(dpo_dataset)} pairs")
print(f"Columns: {dpo_dataset.column_names}")
print("\nLength stats (chars):")
print(f"  Prompt:   avg={sum(prompt_lens)//len(prompt_lens):,}, max={max(prompt_lens):,}")
print(f"  Chosen:   avg={sum(chosen_lens)//len(chosen_lens):,}, max={max(chosen_lens):,}")
print(f"  Rejected: avg={sum(rejected_lens)//len(rejected_lens):,}, max={max(rejected_lens):,}")

sample = formatted_pairs[0]
print("\nSample formatted pair:")
print(f"PROMPT:   {sample['prompt'][:300]}...")
print(f"CHOSEN:   {sample['chosen'][:200]}...")
print(f"REJECTED: {sample['rejected'][:200]}...")

del raw_pairs

## 6. Prepare SFT LoRA for DPO Training

Continue training the loaded SFT adapter directly. The output remains a single LoRA relative to the Qwen3.8 27B base model.

In [ ]:
FastLanguageModel.for_training(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = trainable / total * 100

print("SFT LoRA adapter ready for DPO training")
print(f"  Trainable params: {trainable:,}")
print(f"  Total params:     {total:,}")
print(f"  Trainable pct:    {pct:.4f}%")

## 7. DPO Trainer Setup

Configure TRL's `DPOTrainer` with a low learning rate and an implicit Unsloth reference model.

In [ ]:
from trl import DPOTrainer, DPOConfig
import math

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = math.ceil(len(dpo_dataset) / effective_batch)
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, int(max_steps * WARMUP_RATIO))

print("DPO Training Plan")
print(f"  DPO pairs:       {len(dpo_dataset)}")
print(f"  Effective batch: {effective_batch}")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total steps:     {max_steps}")
print(f"  Warmup steps:    {warmup_steps}")
print(f"  DPO beta:        {DPO_BETA}")

TRAIN_DIR.mkdir(parents=True, exist_ok=True)

if not hasattr(model, "warnings_issued"):
    object.__setattr__(model, "warnings_issued", {})
if hasattr(model, "base_model"):
    if not hasattr(model.base_model, "warnings_issued"):
        object.__setattr__(model.base_model, "warnings_issued", {})
    if hasattr(model.base_model, "model") and not hasattr(model.base_model.model, "warnings_issued"):
        model.base_model.model.warnings_issued = {}

# Belt-and-braces on top of the Processor unwrap in the model cell. Qwen3.8 27B exposes a
# multimodal Conditional-Generation config; if TRL still reads model_type it would mark
# the model vision-capable and route through process_row(), which expects a Processor with
# `.tokenizer`. Our dataset is text-only. Force the text config while DPOTrainer caches
# its routing flags, then restore it.
_orig_model_type = getattr(model.config, "model_type", None)
_text_model_type = None
if hasattr(model.config, "text_config") and model.config.text_config is not None:
    _text_model_type = getattr(model.config.text_config, "model_type", None)

if _text_model_type and _text_model_type != _orig_model_type:
    model.config.model_type = _text_model_type
    print(f"  Temporarily swapped model.config.model_type: {_orig_model_type!r} -> {_text_model_type!r}")
else:
    print(f"  model.config.model_type ({_orig_model_type!r}) left unchanged")

print(f"  processing_class: {type(text_tokenizer).__name__}")

try:
    trainer = DPOTrainer(
        model=model,
        ref_model=None,
        processing_class=text_tokenizer,
        train_dataset=dpo_dataset,
        args=DPOConfig(
            beta=DPO_BETA,
            loss_type=LOSS_TYPE,
            max_length=MAX_SEQ_LENGTH,
            max_prompt_length=MAX_PROMPT_LENGTH,
            # False on purpose: TRL's built-in precompute is one-shot and in-memory. The
            # persistent shard cache cell below does it resumably instead.
            precompute_ref_log_probs=False,
            precompute_ref_batch_size=1,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_steps=warmup_steps,
            max_steps=max_steps,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            optim="adamw_8bit",
            weight_decay=0.0,
            seed=3407,
            gradient_checkpointing=True,
            dataloader_pin_memory=False,
            output_dir=str(TRAIN_DIR),
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            logging_steps=5,
            report_to="none",
            dataset_num_proc=1,
        ),
    )
finally:
    if _orig_model_type is not None:
        model.config.model_type = _orig_model_type

trainer.is_vision_model = False

print("\nDPO Trainer configured")
print(f"  trainer.is_vision_model:  {trainer.is_vision_model}")
print(f"  model.config.model_type:  {model.config.model_type!r}")
print(f"  precompute_ref_log_probs: False (persistent cache cell handles this)")
print(f"  dataloader_pin_memory:    False")
print(f"  Checkpoints:              every {SAVE_STEPS} steps, keep {SAVE_TOTAL_LIMIT}")
print(f"  Precision:                {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

## 8. Persistent Reference Logprob Cache

Precompute and persist frozen-reference log probabilities in resumable shards before DPO training.

TRL's built-in precompute is one-shot and in-memory: a crash three hours in throws away all of it. This writes completed shards to disk with a config fingerprint, so a failed run resumes from the first missing shard instead of from zero.

In [ ]:
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# Persistent, resumable reference logprob cache.
# TRL's built-in precompute is one-shot and in-memory; this saves completed
# shards so a failed run resumes from the first missing shard instead of zero.
import hashlib
import json
import os
import time
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_from_disk

REF_LOGPROBS_CACHE_DIR = TRAIN_DIR / "ref_logprobs_cache"
REF_LOGPROBS_SHARD_DIR = REF_LOGPROBS_CACHE_DIR / "shards"
REF_LOGPROBS_DATASET_DIR = REF_LOGPROBS_CACHE_DIR / "dataset"
REF_LOGPROBS_MANIFEST = REF_LOGPROBS_CACHE_DIR / "manifest.json"
REF_LOGPROBS_SHARD_SIZE = 64
REF_LOGPROBS_BATCH_SIZE = 1

REF_LOGPROBS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REF_LOGPROBS_SHARD_DIR.mkdir(parents=True, exist_ok=True)

_ref_cache_config = {
    "cache_version": 1,
    "base_llm": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "dpo_data_file": str(DPO_DATA_FILE),
    "dpo_data_mtime_ns": DPO_DATA_FILE.stat().st_mtime_ns if DPO_DATA_FILE.exists() else None,
    "dataset_len": len(trainer.train_dataset),
    "dataset_columns": sorted([c for c in trainer.train_dataset.column_names if not c.startswith("ref_")]),
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "tokenizer_class": type(text_tokenizer).__name__,
    "tokenizer_vocab_size": len(text_tokenizer),
    "shard_size": REF_LOGPROBS_SHARD_SIZE,
    "batch_size": REF_LOGPROBS_BATCH_SIZE,
}
_ref_cache_fingerprint = hashlib.sha256(
    json.dumps(_ref_cache_config, sort_keys=True).encode("utf-8")
).hexdigest()

def _read_ref_manifest():
    if not REF_LOGPROBS_MANIFEST.exists():
        return None
    with open(REF_LOGPROBS_MANIFEST) as f:
        return json.load(f)


def _load_valid_ref_shard(shard_path, shard_idx, start, end):
    try:
        shard_payload = torch.load(shard_path, map_location="cpu")
    except (EOFError, OSError, RuntimeError, ValueError) as exc:
        raise RuntimeError(f"Unreadable ref logprob shard: {shard_path}") from exc

    if not isinstance(shard_payload, dict):
        raise RuntimeError(f"Invalid ref logprob shard payload: {shard_path}")
    if shard_payload.get("fingerprint") != _ref_cache_fingerprint:
        raise RuntimeError(f"Stale ref logprob shard fingerprint: {shard_path}")
    if shard_payload.get("shard_idx") != shard_idx:
        raise RuntimeError(f"Ref logprob shard index mismatch: {shard_path}")
    if shard_payload.get("start") != start or shard_payload.get("end") != end:
        raise RuntimeError(f"Ref logprob shard range mismatch: {shard_path}")

    expected_rows = end - start
    for column in ("ref_chosen_logps", "ref_rejected_logps"):
        values = shard_payload.get(column)
        if not isinstance(values, torch.Tensor) or values.numel() != expected_rows:
            raise RuntimeError(f"Invalid {column} values in ref logprob shard: {shard_path}")
    return shard_payload


def _write_ref_manifest(status, completed_shards):
    manifest = {
        "status": status,
        "fingerprint": _ref_cache_fingerprint,
        "config": _ref_cache_config,
        "completed_shards": completed_shards,
        "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp_path = REF_LOGPROBS_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(manifest, f, indent=2)
    os.replace(tmp_path, REF_LOGPROBS_MANIFEST)

_manifest = _read_ref_manifest()
_cache_matches = _manifest is not None and _manifest.get("fingerprint") == _ref_cache_fingerprint
_required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}

if _cache_matches and REF_LOGPROBS_DATASET_DIR.exists():
    cached_dataset = load_from_disk(str(REF_LOGPROBS_DATASET_DIR))
    if len(cached_dataset) == len(trainer.train_dataset) and _required_ref_cols.issubset(cached_dataset.column_names):
        trainer.train_dataset = cached_dataset
        trainer._precomputed_train_ref_log_probs = True
        print(f"Loaded persistent ref logprob cache: {REF_LOGPROBS_DATASET_DIR}")
    else:
        print("Ignoring stale ref logprob dataset cache: length or columns do not match")
        _cache_matches = False

if not _required_ref_cols.issubset(trainer.train_dataset.column_names):
    if not _cache_matches:
        for stale_shard in REF_LOGPROBS_SHARD_DIR.glob("shard-*.pt"):
            stale_shard.unlink()
        _write_ref_manifest("in_progress", [])
        _manifest = _read_ref_manifest()

    num_rows = len(trainer.train_dataset)
    shard_ranges = [
        (start, min(start + REF_LOGPROBS_SHARD_SIZE, num_rows))
        for start in range(0, num_rows, REF_LOGPROBS_SHARD_SIZE)
    ]

    print("Persistent reference logprob cache")
    print(f"  Rows:       {num_rows}")
    print(f"  Shards:     {len(shard_ranges)}")
    print(f"  Shard size: {REF_LOGPROBS_SHARD_SIZE}")
    print(f"  Cache dir:  {REF_LOGPROBS_CACHE_DIR}")

    completed = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if shard_path.exists():
            try:
                _load_valid_ref_shard(shard_path, shard_idx, start, end)
            except RuntimeError as exc:
                print(f"Recomputing corrupt ref logprob shard {shard_idx}: {exc}")
                shard_path.unlink()
            else:
                completed.append(shard_idx)
                continue

        shard_dataset = trainer.train_dataset.select(range(start, end))
        shard_loader = DataLoader(
            shard_dataset,
            batch_size=REF_LOGPROBS_BATCH_SIZE,
            collate_fn=trainer.data_collator,
            num_workers=0,
            pin_memory=False,
            shuffle=False,
        )
        shard_loader = trainer.accelerator.prepare(shard_loader)

        ref_chosen_logps = []
        ref_rejected_logps = []
        for padded_batch in tqdm(shard_loader, desc=f"Ref logprobs shard {shard_idx + 1}/{len(shard_ranges)}"):
            ref_chosen_logp, ref_rejected_logp = trainer.compute_ref_log_probs(padded_batch)
            ref_chosen_logp, ref_rejected_logp = trainer.accelerator.gather_for_metrics(
                (ref_chosen_logp, ref_rejected_logp)
            )
            ref_chosen_logps.append(ref_chosen_logp.float().cpu())
            ref_rejected_logps.append(ref_rejected_logp.float().cpu())
            torch.cuda.empty_cache()
            trainer.accelerator.free_memory()

        shard_payload = {
            "fingerprint": _ref_cache_fingerprint,
            "shard_idx": shard_idx,
            "start": start,
            "end": end,
            "ref_chosen_logps": torch.cat(ref_chosen_logps),
            "ref_rejected_logps": torch.cat(ref_rejected_logps),
        }
        tmp_shard_path = shard_path.with_suffix(".pt.tmp")
        torch.save(shard_payload, tmp_shard_path)
        os.replace(tmp_shard_path, shard_path)
        completed.append(shard_idx)
        _write_ref_manifest("in_progress", completed)

    all_ref_chosen_logps = []
    all_ref_rejected_logps = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if not shard_path.exists():
            raise RuntimeError(f"Missing ref logprob shard: {shard_path}")
        shard_payload = _load_valid_ref_shard(shard_path, shard_idx, start, end)
        all_ref_chosen_logps.append(shard_payload["ref_chosen_logps"])
        all_ref_rejected_logps.append(shard_payload["ref_rejected_logps"])

    ref_chosen_values = torch.cat(all_ref_chosen_logps).numpy()
    ref_rejected_values = torch.cat(all_ref_rejected_logps).numpy()
    if len(ref_chosen_values) != num_rows or len(ref_rejected_values) != num_rows:
        raise RuntimeError("Ref logprob cache length does not match training dataset")

    train_dataset_with_ref = trainer.train_dataset
    for ref_col in ["ref_chosen_logps", "ref_rejected_logps"]:
        if ref_col in train_dataset_with_ref.column_names:
            train_dataset_with_ref = train_dataset_with_ref.remove_columns(ref_col)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_chosen_logps", ref_chosen_values)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_rejected_logps", ref_rejected_values)

    if REF_LOGPROBS_DATASET_DIR.exists():
        import shutil
        shutil.rmtree(REF_LOGPROBS_DATASET_DIR)
    train_dataset_with_ref.save_to_disk(str(REF_LOGPROBS_DATASET_DIR))
    trainer.train_dataset = train_dataset_with_ref
    trainer._precomputed_train_ref_log_probs = True
    _write_ref_manifest("complete", list(range(len(shard_ranges))))
    print(f"Saved persistent ref logprob dataset cache: {REF_LOGPROBS_DATASET_DIR}")

print(f"Ref logprob columns ready: {sorted(_required_ref_cols)}")

## 9. Train

In [ ]:
import gc, os
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint


class PeriodicMemoryCleanup(TrainerCallback):
    """Collect dead objects and return cached CUDA blocks to the allocator.

    Runs every `every` optimizer steps. The previous revision flushed on BOTH
    on_step_begin and on_step_end of EVERY step, which measurably slows training for no
    additional benefit - the allocator only needs periodic relief, and on GB10 host and
    device draw from the same 128 GB pool so the flush is not free.
    """

    def __init__(self, every=50):
        self.every = max(1, int(every))

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
        return control


trainer.add_callback(PeriodicMemoryCleanup(CLEANUP_STEPS))

required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}
if not required_ref_cols.issubset(trainer.train_dataset.column_names):
    raise RuntimeError("Run the persistent reference-logprob cache cell before training.")

print("DPO Training started...")
print(f"  Memory cleanup every {CLEANUP_STEPS} steps")
print("  Watch for loss to decrease from about 0.69 toward 0.40-0.55.")
print("  Reward margins should widen (chosen > rejected).\n")

_ckpt_dir = trainer.args.output_dir
last_ckpt = get_last_checkpoint(_ckpt_dir) if os.path.isdir(_ckpt_dir) else None
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    result = trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("No checkpoint found - starting from scratch.")
    result = trainer.train()

print("\nDPO training complete")
print(f"  Final loss:  {result.training_loss:.4f}")
print(f"  Total steps: {result.global_step}")

## 10. Save DPO LoRA Adapter

Save the combined SFT+DPO LoRA adapter plus the persona system prompts and a completion sentinel. Load this adapter directly on top of `unsloth/Qwen3.8-27B` or compatible Qwen3.8 27B base weights.

In [ ]:
import json
from datetime import datetime, timezone

LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(LORA_OUTPUT_DIR))
# Save the Processor when we unwrapped one - save_pretrained on a Processor writes the
# tokenizer files too, so this is a superset of tokenizer.save_pretrained().
(processor or tokenizer).save_pretrained(str(LORA_OUTPUT_DIR))

prompts_path = LORA_OUTPUT_DIR / "persona_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(persona_system_prompts, f, indent=2)

# Machine-readable completion sentinel (docs/README.md: "Final model-producing notebooks
# must write a machine-readable completion sentinel").
sentinel = {
    "status": "complete",
    "stage": "dpo",
    "model_name": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "sft_adapter": str(SFT_LORA_PATH),
    "input_data_file": str(DPO_DATA_FILE),
    "output_dir": str(TRAIN_DIR),
    "lora_output_dir": str(LORA_OUTPUT_DIR),
    "global_step": int(result.global_step),
    "final_loss": float(result.training_loss),
    "dpo_pairs": int(len(dpo_dataset)),
    "beta": DPO_BETA,
    "loss_type": LOSS_TYPE,
    "learning_rate": LEARNING_RATE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "personas": sorted(persona_system_prompts),
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
with open(LORA_OUTPUT_DIR / "complete.json", "w") as f:
    json.dump(sentinel, f, indent=2)

print(f"DPO LoRA adapter saved to: {LORA_OUTPUT_DIR}")
print(f"Persona prompts saved to:  {prompts_path}")
print(f"Sentinel written:          {LORA_OUTPUT_DIR / 'complete.json'}")

total_size = 0
for path in sorted(LORA_OUTPUT_DIR.iterdir()):
    size = path.stat().st_size
    total_size += size
    print(f"  {path.name:40s} {size / 1024 / 1024:8.1f} MB")
print(f"  {'TOTAL':40s} {total_size / 1024 / 1024:8.1f} MB")

print("\nVerify the adapter scope before shipping:")
print("  docker exec unsloth-notebook python /workspace/training/docs/audit_adapters.py stoic")

## 11. Quick Evaluation

Smoke test a few personas using their extracted system prompts. The DPO model should keep the persona voice while avoiding generic answers.

Uses `ENABLE_THINKING` from Cell 1 so the rendered prompt matches what training saw.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

test_personas = list(persona_system_prompts.keys())[:4]
print(f"Quick evaluation across {len(test_personas)} personas "
      f"(thinking={'ON' if ENABLE_THINKING else 'OFF'})\n")

for persona_key in test_personas:
    system_prompt = persona_system_prompts[persona_key]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    # enable_thinking must be passed explicitly. Omitting it is NOT the same as False:
    # the template defaults to thinking ON, which prepends Qwen3.8's xhigh reasoning
    # instruction to the persona system prompt and opens an unclosed <think> block.
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )
    inputs = tokenizer(text=text, return_tensors="pt").to(model.device)

    print("=" * 70)
    print(f"PERSONA: {persona_key.upper()}")
    print(f"Q: {TEST_PROMPT}")
    print("A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024 if ENABLE_THINKING else 256,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        top_k=GEN_TOP_K,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print("\n")

del inputs, outputs

## 12. Verify Adapter Reload

Reload the saved DPO adapter from disk to confirm it is self-contained and portable.

In [ ]:
import gc

del model, tokenizer, trainer, dpo_dataset
gc.collect()
torch.cuda.empty_cache()

print(f"Reloading adapter from: {LORA_OUTPUT_DIR}")
model2, tokenizer2 = FastLanguageModel.from_pretrained(
    str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model2)

if hasattr(tokenizer2, "tokenizer"):
    tokenizer2 = tokenizer2.tokenizer

with open(LORA_OUTPUT_DIR / "persona_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

test_key = list(reloaded_prompts.keys())[0]
messages = [
    {"role": "system", "content": reloaded_prompts[test_key]},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)
inputs = tokenizer2(text=text, return_tensors="pt").to(model2.device)
outputs = model2.generate(
    **inputs,
    max_new_tokens=1024 if ENABLE_THINKING else 256,
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    top_k=GEN_TOP_K,
    do_sample=True,
)
response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"Adapter reload test persona: {test_key}")
print(f"Q: {TEST_PROMPT}")
print(f"A: {response[:500]}")
print("\nAdapter reload test complete - ready for deployment via vLLM.")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()

## 13. Export Merged Model to GGUF (Mobile / Cross-Platform) — optional

Merge the SFT+DPO LoRA adapter into the base Qwen3.8 27B and export to GGUF so it can run
anywhere llama.cpp / Ollama / LM Studio / iOS GGUF runners are supported.

- `q4_k_m` is the recommended quant for phones (best size/quality tradeoff).
- `q5_k_m` and `q8_0` are also produced for higher-quality desktop use.
- Requires Unsloth's bundled `llama.cpp` build (downloaded automatically on first call).

**Not needed for the vLLM deployment path** — the LoRA adapter saved in Cell 10 is the
shipping artifact. Skip these cells unless you specifically want GGUF.

**Memory:** this reloads the base at `load_in_4bit=False`, i.e. ~56 GB of bf16 weights,
then merges. Run it in a fresh kernel if training left the pool fragmented.

In [ ]:
# Reload adapter fresh so we export from a clean state
import gc, torch
from pathlib import Path
from unsloth import FastLanguageModel

try:
    del model, tokenizer, trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

GGUF_OUTPUT_DIR = OUTPUT_BASE_DIR / "gguf"
GGUF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load base + adapter; Unsloth's GGUF exporter will merge before conversion
model_gguf, tokenizer_gguf = FastLanguageModel.from_pretrained(
    model_name=str(LORA_OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,   # full-precision merge for accurate GGUF quantization
)

# Force the chat-template's end-of-turn token as the GGUF EOS.
#
# Qwen ChatML uses `<|im_end|>` as the closer of `<|im_start|>role\n...content...<|im_end|>\n`.
# Without forcing this, llama.cpp's GGUF converter may inherit a generic `<eos>`
# from `text_config.eos_token_id` on multimodal Conditional-Generation models,
# so the model never stops on `<|im_end|>` and emits the next turn header.
# iOS GGUF runners then strip the leading `<|im_start|>` special token and
# render the bare role text ("user\n...") plus a hallucinated user question.
_render = tokenizer_gguf.apply_chat_template(
    [{"role": "user", "content": "x"}, {"role": "assistant", "content": "y"}],
    tokenize=False,
)
_eot_candidates = ["<|im_end|>", "<turn|>", "<end_of_turn>", "<|eot_id|>"]
eot_token = next((c for c in _eot_candidates if c in _render), None)
if eot_token is None:
    raise RuntimeError(f"Could not detect end-of-turn marker in chat template. Rendered: {_render!r}")
# Multimodal processors wrap the tokenizer; unwrap so we can call tokenizer methods.
_tok = getattr(tokenizer_gguf, "tokenizer", tokenizer_gguf)
eot_id = _tok.convert_tokens_to_ids(eot_token)
if eot_id is None or eot_id == _tok.unk_token_id:
    raise RuntimeError(f"{eot_token!r} not in tokenizer vocab (got id={eot_id})")

_tok.eos_token = eot_token
model_gguf.config.eos_token_id = eot_id
# Qwen3.8 27B may be a Conditional-Generation model with a nested text_config;
# the llama.cpp converter reads from there, so we MUST set it here too.
if hasattr(model_gguf.config, "text_config") and model_gguf.config.text_config is not None:
    model_gguf.config.text_config.eos_token_id = eot_id
# generation_config.eos_token_id may be a list; collapse to the single
# chat-template EOS so runners that only honor a scalar pick the right one.
if getattr(model_gguf, "generation_config", None) is not None:
    model_gguf.generation_config.eos_token_id = eot_id

print(f"\u2713 Forced GGUF EOS to {eot_token!r} (id={eot_id})")

# Pass ALL quant methods in one call so the LoRA→FP16 merge happens ONCE
# and llama.cpp quantizes from that single merged file. Otherwise the loop
# re-merges and re-writes the FP16 GGUF for every quant level.
QUANT_METHODS = ["q4_k_m", "q5_k_m", "q8_0"]

print(f"Exporting GGUF (single merge → {len(QUANT_METHODS)} quants)...")
model_gguf.save_pretrained_gguf(
    str(GGUF_OUTPUT_DIR),
    tokenizer_gguf,
    quantization_method=QUANT_METHODS,
)

print(f"\n\u2713 GGUF export complete: {GGUF_OUTPUT_DIR}")
for f in sorted(GGUF_OUTPUT_DIR.glob("*.gguf")):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:60s} {size_mb:>8.1f} MB")

print("\nMobile usage (iPhone):")
print("  1. Transfer the q4_k_m .gguf file to your phone (AirDrop / Files app).")
print("  2. Open in an iOS GGUF runner (LLMFarm, PocketPal, Private LLM, etc.).")
print("  3. Use Qwen ChatML template; set context length <= MAX_SEQ_LENGTH.")

del model_gguf, tokenizer_gguf
gc.collect()
torch.cuda.empty_cache()